<a href="https://colab.research.google.com/github/pradervonsky/vbig-lab/blob/main/evaluation/iaa-validation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Inter-Annotator Agreement (IAA)

Pipeline:
1. Pull `human_insights` rows from Supabase
2. Build units, parsing the dashboard-chart-level LoD
3. Compute BERTScore-based pairwise distances for all 3 annotator pairs
4. Compute Krippendorff's α overall, per level, and per dashboard
5. Compute ROUGE, BLEU, METEOR, BERTScore

## Initial steps

In [1]:
!pip install -q bert-score supabase pandas numpy openpyxl rouge_score

In [2]:
import pandas as pd
import numpy as np
import re
import torch
import nltk
from itertools import combinations
from bert_score import BERTScorer
from bert_score import score as bert_score
from supabase import create_client
from google.colab import userdata
from rouge_score import rouge_scorer
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score

In [3]:
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('wordnet')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

### Pull data from Supabase

In [4]:
SUPABASE_URL = userdata.get("SUPABASE_URL")
SUPABASE_KEY = userdata.get("SUPABASE_KEY")

supabase = create_client(SUPABASE_URL, SUPABASE_KEY)
response = supabase.table("human_insights").select("*").execute()
df_raw = pd.DataFrame(response.data)

print("irr_flag unique values:", df_raw["irr_flag"].unique())
irr_df = df_raw[df_raw["irr_flag"] == True].reset_index(drop=True)
print(f"IRR rows: {len(irr_df)}")

irr_flag unique values: [False  True]
IRR rows: 10


### Parsing step

In [5]:
NA_PATTERN = re.compile(r"^(not applicable|n/a)$", re.IGNORECASE)

def is_missing(text):
    if text is None:
        return True
    # Strip whitespace and trailing punctuation before matching
    cleaned = str(text).strip().rstrip(".")
    cleaned = cleaned.strip()  # strip again after removing dot
    return cleaned.lower() in {"not applicable", "n/a", ""}

def normalize_value(val):
    if val is None:
        return None
    cleaned = str(val).strip().rstrip(".").strip()
    if cleaned == "" or NA_PATTERN.match(cleaned):
        return None
    return cleaned

def parse_charts(text):
    charts = []
    if not text or (isinstance(text, float) and np.isnan(text)):
        return charts

    blocks = re.split(r"(?=Chart\s+\d+\s*[:.])", text.strip())
    for block in blocks:
        block = block.strip()
        if not block:
            continue
        lines = block.splitlines()
        header = lines[0].strip()
        match = re.match(r"Chart\s+(\d+)\s*[:.]\s*(.*)", header)
        if not match:
            continue
        chart_id = int(match.group(1))
        title    = match.group(2).strip()

        L2 = L3 = L4 = None
        for line in lines[1:]:
            line = line.strip()
            if line.startswith("L2:"):
                L2 = normalize_value(line[3:].strip())
            elif line.startswith("L3:"):
                L3 = normalize_value(line[3:].strip())
            elif line.startswith("L4:"):
                L4 = normalize_value(line[3:].strip())

        charts.append({
            "chart_id": chart_id,
            "title":    title,
            "L2":       L2,
            "L3":       L3,
            "L4":       L4,
        })
    return charts

In [6]:
ANNOTATOR_COLS = ["insight_part_1", "insight_part_2", "insight_part_3"]

records = []
for _, row in irr_df.iterrows():
    row_id      = row["id"]
    metadata_id = row["metadata_id"]

    # Parse each annotator's blob
    parsed = {col: {c["chart_id"]: c for c in parse_charts(row[col])}
              for col in ANNOTATOR_COLS}

    # All chart_ids seen across any annotator
    all_chart_ids = sorted(
        set().union(*[set(p.keys()) for p in parsed.values()])
    )

    for chart_id in all_chart_ids:
        # Get title from whichever annotator has this chart
        title = next(
            (parsed[col][chart_id]["title"]
             for col in ANNOTATOR_COLS
             if chart_id in parsed[col]),
            ""
        )
        for level in ["L2", "L3", "L4"]:
            records.append({
                "row_id":      row_id,
                "metadata_id": metadata_id,
                "chart_id":    chart_id,
                "title":       title,
                "level":       level,
                "unit_id":     f"{row_id}__chart{chart_id}__{level}",
                "annotator_1": parsed["insight_part_1"].get(chart_id, {}).get(level),
                "annotator_2": parsed["insight_part_2"].get(chart_id, {}).get(level),
                "annotator_3": parsed["insight_part_3"].get(chart_id, {}).get(level),
            })

long_df = pd.DataFrame(records)
print(f"\nTotal units: {len(long_df)}")
print(long_df.head(12).to_string())


Total units: 147
                                  row_id                           metadata_id  chart_id                        title level                                           unit_id                                                                                                                                                                                                                                                                                             annotator_1                                                                                                                                                                                                                                                                                                              annotator_2                                                                                                                                                                                                        

In [7]:
print("\nNOT APPLICABLE counts per level per annotator:")
for level in ["L2", "L3", "L4"]:
    sub = long_df[long_df["level"] == level]
    print(f"  {level}: "
          f"ann1={sub['annotator_1'].isna().sum()} | "
          f"ann2={sub['annotator_2'].isna().sum()} | "
          f"ann3={sub['annotator_3'].isna().sum()} | "
          f"total_units={len(sub)}")


NOT APPLICABLE counts per level per annotator:
  L2: ann1=0 | ann2=0 | ann3=0 | total_units=49
  L3: ann1=14 | ann2=4 | ann3=2 | total_units=49
  L4: ann1=0 | ann2=0 | ann3=0 | total_units=49


In [8]:
# Check chart count per dashboard per annotator
print("=== Chart count per dashboard per annotator ===\n")

for row_id in sorted(long_df["row_id"].unique()):
    sub = long_df[long_df["row_id"] == row_id]

    # Get unique charts seen per annotator
    ann1_charts = sub[sub["annotator_1"].notna()]["chart_id"].unique()
    ann2_charts = sub[sub["annotator_2"].notna()]["chart_id"].unique()
    ann3_charts = sub[sub["annotator_3"].notna()]["chart_id"].unique()

    total_charts = sub["chart_id"].nunique()
    metadata_id = sub["metadata_id"].iloc[0]

    print(f"Dashboard: {row_id[:8]}… (metadata: {metadata_id[:8]}…)")
    print(f"  Total charts parsed: {total_charts}")
    print(f"  ann1 charts with content: {sorted(ann1_charts)} ({len(ann1_charts)})")
    print(f"  ann2 charts with content: {sorted(ann2_charts)} ({len(ann2_charts)})")
    print(f"  ann3 charts with content: {sorted(ann3_charts)} ({len(ann3_charts)})")

    # Flag mismatches
    all_charts = set(sub["chart_id"].unique())
    for ann_label, ann_charts in [("ann1", ann1_charts), ("ann2", ann2_charts), ("ann3", ann3_charts)]:
        missing = all_charts - set(ann_charts)
        if missing:
            print(f"  ⚠️  {ann_label} missing charts: {sorted(missing)}")
    print()

=== Chart count per dashboard per annotator ===

Dashboard: 08006d53… (metadata: 27052e58…)
  Total charts parsed: 4
  ann1 charts with content: [np.int64(1), np.int64(2), np.int64(3), np.int64(4)] (4)
  ann2 charts with content: [np.int64(1), np.int64(2), np.int64(3), np.int64(4)] (4)
  ann3 charts with content: [np.int64(1), np.int64(2), np.int64(3), np.int64(4)] (4)

Dashboard: 1120a289… (metadata: 4f4b551b…)
  Total charts parsed: 5
  ann1 charts with content: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)] (5)
  ann2 charts with content: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)] (5)
  ann3 charts with content: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)] (5)

Dashboard: 2e5b2881… (metadata: 932c11c0…)
  Total charts parsed: 6
  ann1 charts with content: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6)] (6)
  ann2 charts with content: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.i

### BERTScore-based distance function

`distance(a, b) = 1 - BERTScore_F1(a, b)`  
`NOT APPLICABLE` entries are treated as missing (`np.nan`).

In [9]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_TYPE = "roberta-large"
print(f"Using device: {DEVICE}, model: {MODEL_TYPE}")

SCORER = BERTScorer(
    model_type=MODEL_TYPE,
    lang="en",
    rescale_with_baseline=True, # interpretable output
    device=DEVICE
)

ANNOTATOR_COLS = ["annotator_1", "annotator_2", "annotator_3"]
ANNOTATOR_PAIRS = list(combinations(ANNOTATOR_COLS, 2))

Using device: cuda, model: roberta-large


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [10]:
def is_missing(text):
    if text is None:
        return True
    return str(text).strip().lower() in {"not applicable", "n/a", ""}

def bertscore_distance_batch(refs, hyps):
    assert len(refs) == len(hyps)
    distances = np.full(len(refs), np.nan)

    valid_indices = [
        i for i, (r, h) in enumerate(zip(refs, hyps))
        if not is_missing(r) and not is_missing(h)
    ]
    if not valid_indices:
        return distances

    valid_refs = [str(refs[i]) for i in valid_indices]
    valid_hyps = [str(hyps[i]) for i in valid_indices]

    _, _, F1 = SCORER.score(valid_hyps, valid_refs)

    for idx, f1_val in zip(valid_indices, F1.numpy()):
        # Clamp F1 to [0,1] before converting to distance
        # rescaled BERTScore can be negative for very dissimilar pairs
        f1_clamped = float(np.clip(f1_val, 0.0, 1.0))
        distances[idx] = 1.0 - f1_clamped

    return distances

In [11]:
# Compute pairwise distances on long_df
DIST_COLS = []
for (col_a, col_b) in ANNOTATOR_PAIRS:
    suffix_a = col_a.split("_")[-1]
    suffix_b = col_b.split("_")[-1]
    pair_key = f"dist_{suffix_a}{suffix_b}"
    DIST_COLS.append(pair_key)
    print(f"Computing {pair_key} ({col_a} vs {col_b}) ...")
    long_df[pair_key] = bertscore_distance_batch(
        long_df[col_a].tolist(),
        long_df[col_b].tolist(),
    )

print("\nSample distances:")
print(long_df[["unit_id", "chart_id", "level"] + DIST_COLS].head(12).to_string(index=False))

Computing dist_12 (annotator_1 vs annotator_2) ...
Computing dist_13 (annotator_1 vs annotator_3) ...
Computing dist_23 (annotator_2 vs annotator_3) ...

Sample distances:
                                         unit_id  chart_id level  dist_12  dist_13  dist_23
f451d481-b828-40ed-a9a4-fc7fd6d6caf2__chart1__L2         1    L2 0.352416 0.456545 0.332972
f451d481-b828-40ed-a9a4-fc7fd6d6caf2__chart1__L3         1    L3      NaN      NaN      NaN
f451d481-b828-40ed-a9a4-fc7fd6d6caf2__chart1__L4         1    L4 0.765398 0.805485 0.751733
f451d481-b828-40ed-a9a4-fc7fd6d6caf2__chart2__L2         2    L2 0.564897 0.756618 0.547600
f451d481-b828-40ed-a9a4-fc7fd6d6caf2__chart2__L3         2    L3 0.779187 0.838229 0.740927
f451d481-b828-40ed-a9a4-fc7fd6d6caf2__chart2__L4         2    L4 0.775373 0.827931 0.611633
f451d481-b828-40ed-a9a4-fc7fd6d6caf2__chart3__L2         3    L2 0.445607 0.496994 0.385439
f451d481-b828-40ed-a9a4-fc7fd6d6caf2__chart3__L3         3    L3 0.669904 0.822849 0.692366


### Krippendorff's alpha with custom BERTScore distance

$$\alpha = 1 - \frac{D_o}{D_e}$$

- **D_o** = mean observed disagreement (average pairwise distance within units, >=2 valid annotators)
- **D_e** = mean expected disagreement (average over all valid distances in the pool)

In [12]:
def compute_cross_item_distances_batched(sub_df, ann_col, device=DEVICE):
    annotations = sub_df[ann_col].tolist()
    n = len(annotations)

    refs_batch = []
    hyps_batch = []

    for i in range(n):
        for j in range(i + 1, n):
            a, b = annotations[i], annotations[j]
            if is_missing(a) or is_missing(b):
                continue
            refs_batch.append(str(a))
            hyps_batch.append(str(b))

    if not refs_batch:
        return []

    _, _, F1 = SCORER.score(hyps_batch, refs_batch)

    # Clamp to [0,1] before converting to distance
    F1_clamped = np.clip(F1.numpy(), 0.0, 1.0)
    return (1.0 - F1_clamped).tolist()


def krippendorff_alpha_bertscore_correct(sub_df, dist_cols=DIST_COLS):
    unit_mean_dist = sub_df[dist_cols].mean(axis=1, skipna=True)
    valid_units    = unit_mean_dist.notna()
    n_valid        = valid_units.sum()

    if n_valid < 2:
        return np.nan, np.nan, np.nan, n_valid

    D_o = unit_mean_dist[valid_units].mean()

    cross_item_distances = []
    for ann_col in ["annotator_1", "annotator_2", "annotator_3"]:
        print(f"  [{ann_col}] computing cross-item distances "
              f"({sub_df[ann_col].notna().sum()} valid annotations)...")
        cross_item_distances.extend(
            compute_cross_item_distances_batched(sub_df, ann_col)
        )

    if not cross_item_distances:
        return np.nan, np.nan, np.nan, n_valid

    D_e = np.mean(cross_item_distances)
    if D_e == 0:
        return np.nan, np.nan, np.nan, n_valid

    alpha = 1.0 - (D_o / D_e)
    return alpha, D_o, D_e, n_valid

## Overall alpha (all dashboards combined)

In [13]:
print("=" * 60)
print(f"Krippendorff's α — BERTScore ({MODEL_TYPE})")
print("=" * 60)

print("\n[Overall]")
alpha_all, Do_all, De_all, n_all = krippendorff_alpha_bertscore_correct(long_df)
print(f"  α={alpha_all:.4f}  D_o={Do_all:.4f}  D_e={De_all:.4f}  n_valid={n_all}")

Krippendorff's α — BERTScore (roberta-large)

[Overall]
  [annotator_1] computing cross-item distances (133 valid annotations)...
  [annotator_2] computing cross-item distances (143 valid annotations)...
  [annotator_3] computing cross-item distances (145 valid annotations)...
  α=0.2211  D_o=0.7119  D_e=0.9140  n_valid=143


## Alpha per insight level (L2 / L3 / L4)

In [14]:
print("\n[By Semantic Level]")
level_results = []
for level in ["L2", "L3", "L4"]:
    print(f"\n  Level {level}:")
    sub = long_df[long_df["level"] == level]
    alpha, D_o, D_e, n_valid = krippendorff_alpha_bertscore_correct(sub)
    level_results.append({
        "Level":         level,
        "Alpha":         round(alpha, 4) if not np.isnan(alpha) else "N/A",
        "D_o":           round(D_o, 4)   if not np.isnan(D_o)   else "N/A",
        "D_e":           round(D_e, 4)   if not np.isnan(D_e)   else "N/A",
        "N_units_valid": n_valid,
        "N_units_total": len(sub),
    })
    print(f"  → α={alpha:.4f}  D_o={D_o:.4f}  D_e={D_e:.4f}  "
          f"valid={n_valid}/{len(sub)}")

level_df = pd.DataFrame(level_results)


[By Semantic Level]

  Level L2:
  [annotator_1] computing cross-item distances (49 valid annotations)...
  [annotator_2] computing cross-item distances (49 valid annotations)...
  [annotator_3] computing cross-item distances (49 valid annotations)...
  → α=0.3388  D_o=0.5753  D_e=0.8700  valid=49/49

  Level L3:
  [annotator_1] computing cross-item distances (35 valid annotations)...
  [annotator_2] computing cross-item distances (45 valid annotations)...
  [annotator_3] computing cross-item distances (47 valid annotations)...
  → α=0.0968  D_o=0.7660  D_e=0.8481  valid=45/49

  Level L4:
  [annotator_1] computing cross-item distances (49 valid annotations)...
  [annotator_2] computing cross-item distances (49 valid annotations)...
  [annotator_3] computing cross-item distances (49 valid annotations)...
  → α=0.0378  D_o=0.7990  D_e=0.8304  valid=49/49


## Alpha per dashboard

In [15]:
print("\n[By Dashboard]")
dash_results = []
for row_id in sorted(long_df["row_id"].unique()):
    print(f"\n  Dashboard {row_id[:8]}…:")
    sub = long_df[long_df["row_id"] == row_id]
    alpha, D_o, D_e, n_valid = krippendorff_alpha_bertscore_correct(sub)
    dash_results.append({
        "row_id":        row_id,
        "Alpha":         round(alpha, 4) if not np.isnan(alpha) else "N/A",
        "D_o":           round(D_o, 4)   if not np.isnan(D_o)   else "N/A",
        "D_e":           round(D_e, 4)   if not np.isnan(D_e)   else "N/A",
        "N_units_valid": n_valid,
        "N_units_total": len(sub),
    })
    print(f"  → α={alpha:.4f}  valid={n_valid}/{len(sub)}")

dash_df = pd.DataFrame(dash_results)


[By Dashboard]

  Dashboard 08006d53…:
  [annotator_1] computing cross-item distances (9 valid annotations)...
  [annotator_2] computing cross-item distances (12 valid annotations)...
  [annotator_3] computing cross-item distances (12 valid annotations)...
  → α=0.2426  valid=12/12

  Dashboard 1120a289…:
  [annotator_1] computing cross-item distances (13 valid annotations)...
  [annotator_2] computing cross-item distances (14 valid annotations)...
  [annotator_3] computing cross-item distances (14 valid annotations)...
  → α=0.2516  valid=14/15

  Dashboard 2e5b2881…:
  [annotator_1] computing cross-item distances (16 valid annotations)...
  [annotator_2] computing cross-item distances (18 valid annotations)...
  [annotator_3] computing cross-item distances (18 valid annotations)...
  → α=0.1831  valid=18/18

  Dashboard 511bca34…:
  [annotator_1] computing cross-item distances (9 valid annotations)...
  [annotator_2] computing cross-item distances (9 valid annotations)...
  [annotat

## Alpha Summary

In [16]:
print("\n[Level Summary]")
print(level_df.to_string(index=False))

print("\n[Dashboard Summary]")
print(dash_df.to_string(index=False))


[Level Summary]
Level  Alpha    D_o    D_e  N_units_valid  N_units_total
   L2 0.3388 0.5753 0.8700             49             49
   L3 0.0968 0.7660 0.8481             45             49
   L4 0.0378 0.7990 0.8304             49             49

[Dashboard Summary]
                              row_id  Alpha    D_o    D_e  N_units_valid  N_units_total
08006d53-f01b-4eb6-a517-8825c184d732 0.2426 0.6484 0.8561             12             12
1120a289-6f8e-4498-91d8-59bb0b7bb49d 0.2516 0.6823 0.9117             14             15
2e5b2881-e3e3-421f-a3b9-098e326303a3 0.1831 0.7137 0.8737             18             18
511bca34-e364-49d6-8cf9-263c3426db13 0.1632 0.7792 0.9311              9              9
723340b5-2604-49f0-963a-3699aef72048 0.1279 0.7337 0.8413             12             12
b164ac02-5f8b-4b10-ad35-afa018951888 0.1809 0.7462 0.9110             18             18
cfe29540-1433-470e-9dc8-612e018bd8b6 0.2178 0.6974 0.8915             20             21
dc09d834-178b-48e2-990d-944644

## Other Metrics

### BERTScore Metric

In [17]:
F1_COLS = {
    "dist_12": "F1_ann1_ann2",
    "dist_13": "F1_ann1_ann3",
    "dist_23": "F1_ann2_ann3",
}

for dist_col, f1_col in F1_COLS.items():
    long_df[f1_col] = 1.0 - long_df[dist_col]  # NaN stays NaN

F1_VALUE_COLS = list(F1_COLS.values())

print("--- Mean Pairwise BERTScore F1 by Level ---")
iaa_results = []
for level in ["L2", "L3", "L4"]:
    sub = long_df[long_df["level"] == level]
    row = {"Level": level}
    for f1_col in F1_VALUE_COLS:
        mean_f1 = sub[f1_col].mean(skipna=True)
        n_valid  = sub[f1_col].notna().sum()
        row[f1_col]               = round(mean_f1, 4)
        row[f1_col + "_n_valid"]  = n_valid
    row["Mean_F1_overall"] = round(
        sub[F1_VALUE_COLS].values.flatten()[
            ~np.isnan(sub[F1_VALUE_COLS].values.flatten())
        ].mean(), 4
    )
    iaa_results.append(row)
    print(f"  {level}: "
          f"ann1-ann2={row['F1_ann1_ann2']:.4f} (n={row['F1_ann1_ann2_n_valid']})  "
          f"ann1-ann3={row['F1_ann1_ann3']:.4f} (n={row['F1_ann1_ann3_n_valid']})  "
          f"ann2-ann3={row['F1_ann2_ann3']:.4f} (n={row['F1_ann2_ann3_n_valid']})  "
          f"mean={row['Mean_F1_overall']:.4f}")

iaa_df = pd.DataFrame(iaa_results)

--- Mean Pairwise BERTScore F1 by Level ---
  L2: ann1-ann2=0.4274 (n=49)  ann1-ann3=0.4341 (n=49)  ann2-ann3=0.4127 (n=49)  mean=0.4247
  L3: ann1-ann2=0.2683 (n=35)  ann1-ann3=0.2003 (n=35)  ann2-ann3=0.2301 (n=45)  mean=0.2327
  L4: ann1-ann2=0.1763 (n=49)  ann1-ann3=0.1725 (n=49)  ann2-ann3=0.2543 (n=49)  mean=0.2010


#### Qualitative analysis: lowest scores

In [18]:
print("=== Lowest Pairwise BERTScore F1 Units ===\n")

N_BOTTOM = 3

for level in ["L2", "L3", "L4"]:
    sub = long_df[long_df["level"] == level].copy()

    # Mean F1 across available pairs per unit
    sub["mean_F1"] = sub[F1_VALUE_COLS].mean(axis=1, skipna=True)

    # Only units with at least one valid pair
    sub_valid = sub[sub["mean_F1"].notna()].sort_values("mean_F1")

    print(f"{'='*70}")
    print(f"LEVEL {level} — Bottom {N_BOTTOM} units by mean pairwise F1")
    print(f"{'='*70}")

    for _, row in sub_valid.head(N_BOTTOM).iterrows():
        print(f"\nunit : {row['unit_id']}")
        print(f"chart: {row['chart_id']} — {row['title']}")
        print(f"F1   : ann1-ann2={row['F1_ann1_ann2']:.4f}  "
              f"ann1-ann3={row['F1_ann1_ann3']:.4f}  "
              f"ann2-ann3={row['F1_ann2_ann3']:.4f}  "
              f"mean={row['mean_F1']:.4f}")
        print(f"ann1 : {row['annotator_1']}")
        print(f"ann2 : {row['annotator_2']}")
        print(f"ann3 : {row['annotator_3']}")
        print()

=== Lowest Pairwise BERTScore F1 Units ===

LEVEL L2 — Bottom 3 units by mean pairwise F1

unit : 511bca34-e364-49d6-8cf9-263c3426db13__chart2__L2
chart: 2 — Profit Margin by State
F1   : ann1-ann2=0.0000  ann1-ann3=0.0000  ann2-ann3=0.0469  mean=0.0156
ann1 : The majority of states show a positive margin of green encoded in 2021, compared to 2020
ann2 : AK, WY, HI, and ME had N/A value
ann3 : States with negatives prorit margin are OR, Az, Co, IL, Tx, TN, OH, PA and NC while others in green has positive profit margin


unit : b164ac02-5f8b-4b10-ad35-afa018951888__chart2__L2
chart: 2 — 2023 | Sales vs Targets
F1   : ann1-ann2=0.1226  ann1-ann3=0.1753  ann2-ann3=0.0073  mean=0.1017
ann1 : Sales started at around £40K in January and ended the year at around £80K, but it did not reach the target
ann2 : In 2023, the sales vs target, J is 21.7K, F is -7.3K, and M is 3.2K
ann3 : January, June, August, October, and November were months with sales above target


unit : dc09d834-178b-48e2-990d-

#### Qualitative analysis: highest scores

In [19]:
print("\n=== Highest Pairwise BERTScore F1 Units ===\n")

N_TOP = 3

for level in ["L2", "L3", "L4"]:
    sub = long_df[long_df["level"] == level].copy()
    sub["mean_F1"] = sub[F1_VALUE_COLS].mean(axis=1, skipna=True)
    sub_valid = sub[sub["mean_F1"].notna()].sort_values("mean_F1", ascending=False)

    print(f"{'='*70}")
    print(f"LEVEL {level} — Top {N_TOP} units by mean pairwise F1")
    print(f"{'='*70}")

    for _, row in sub_valid.head(N_TOP).iterrows():
        print(f"\nunit : {row['unit_id']}")
        print(f"chart: {row['chart_id']} — {row['title']}")
        print(f"F1   : ann1-ann2={row['F1_ann1_ann2']:.4f}  "
              f"ann1-ann3={row['F1_ann1_ann3']:.4f}  "
              f"ann2-ann3={row['F1_ann2_ann3']:.4f}  "
              f"mean={row['mean_F1']:.4f}")
        print(f"ann1 : {row['annotator_1']}")
        print(f"ann2 : {row['annotator_2']}")
        print(f"ann3 : {row['annotator_3']}")
        print()


=== Highest Pairwise BERTScore F1 Units ===

LEVEL L2 — Top 3 units by mean pairwise F1

unit : 1120a289-6f8e-4498-91d8-59bb0b7bb49d__chart1__L2
chart: 1 — Scoreboard Overview
F1   : ann1-ann2=0.8352  ann1-ann3=0.6942  ann2-ann3=0.6599  mean=0.7298
ann1 : Sales are 733,215, profit is 93,439, and the number of orders is 1,687
ann2 : Sales are 733,215, profit is 93.439, and orders are 1,687
ann3 : Sales value is 733,215, Profit value is 93,439, and Orders values is 1,687


unit : 08006d53-f01b-4eb6-a517-8825c184d732__chart1__L2
chart: 1 — Scoreboard & Barchart
F1   : ann1-ann2=0.8238  ann1-ann3=0.6703  ann2-ann3=0.6409  mean=0.7117
ann1 : Total sales are $745.6K, total orders are $3.4K, and sales are $5.4K above the target
ann2 : Total sales are $745.6K, total number of orders is 3.4K, and the sales are $5.4 above target
ann3 : Total sales is 745.6K, Total Orders is 3.4K and Sales is above target with 5.4K in value


unit : cfe29540-1433-470e-9dc8-612e018bd8b6__chart1__L2
chart: 1 — Sco

#### Distribution summary per level

In [20]:
print("\n=== F1 Distribution Summary per Level ===\n")
for level in ["L2", "L3", "L4"]:
    sub = long_df[long_df["level"] == level].copy()
    sub["mean_F1"] = sub[F1_VALUE_COLS].mean(axis=1, skipna=True)
    vals = sub["mean_F1"].dropna()
    print(f"  {level}: min={vals.min():.4f}  "
          f"Q1={vals.quantile(0.25):.4f}  "
          f"median={vals.quantile(0.50):.4f}  "
          f"Q3={vals.quantile(0.75):.4f}  "
          f"max={vals.max():.4f}")


=== F1 Distribution Summary per Level ===

  L2: min=0.0156  Q1=0.3050  median=0.3939  Q3=0.5678  max=0.7298
  L3: min=0.0027  Q1=0.1867  median=0.2461  Q3=0.2954  max=0.4646
  L4: min=0.0794  Q1=0.1604  median=0.2045  Q3=0.2471  max=0.3028


### ROUGE Metric

In [21]:
scorer = rouge_scorer.RougeScorer(
    ["rouge1", "rouge2", "rougeL"],
    use_stemmer=True
)

# Pairwise ROUGE F1 between two text lists
def rouge_pairwise_batch(refs, hyps):
    assert len(refs) == len(hyps)

    results = {
        "rouge1": np.full(len(refs), np.nan),
        "rouge2": np.full(len(refs), np.nan),
        "rougeL": np.full(len(refs), np.nan),
    }

    for i, (r, h) in enumerate(zip(refs, hyps)):
        if is_missing(r) or is_missing(h):
            continue
        scores = scorer.score(str(r), str(h))
        results["rouge1"][i] = scores["rouge1"].fmeasure
        results["rouge2"][i] = scores["rouge2"].fmeasure
        results["rougeL"][i] = scores["rougeL"].fmeasure

    return results

# Compute pairwise ROUGE for all annotator pairs
ROUGE_METRICS = ["rouge1", "rouge2", "rougeL"]
ROUGE_PAIR_COLS = {}  # maps (pair_key, metric) -> column name

for (col_a, col_b) in ANNOTATOR_PAIRS:
    suffix_a = col_a.split("_")[-1]
    suffix_b = col_b.split("_")[-1]
    pair_key = f"{suffix_a}{suffix_b}"

    print(f"Computing ROUGE for ann{suffix_a} vs ann{suffix_b} ...")
    results = rouge_pairwise_batch(
        long_df[col_a].tolist(),
        long_df[col_b].tolist()
    )

    for metric in ROUGE_METRICS:
        col_name = f"rouge_{metric}_{pair_key}"
        long_df[col_name] = results[metric]
        ROUGE_PAIR_COLS[(pair_key, metric)] = col_name

print("\nDone. Sample:")
sample_cols = ["unit_id", "level"] + list(ROUGE_PAIR_COLS.values())[:6]
print(long_df[sample_cols].head(6).to_string(index=False))

# Mean pairwise ROUGE F1 per level
print("\n=== Mean Pairwise ROUGE F1 by Level ===\n")

rouge_iaa_results = []
for level in ["L2", "L3", "L4"]:
    sub = long_df[long_df["level"] == level]
    row = {"Level": level}

    for metric in ROUGE_METRICS:
        pair_scores = []
        pair_details = []

        for (col_a, col_b) in ANNOTATOR_PAIRS:
            suffix_a = col_a.split("_")[-1]
            suffix_b = col_b.split("_")[-1]
            pair_key = f"{suffix_a}{suffix_b}"
            col_name = ROUGE_PAIR_COLS[(pair_key, metric)]

            mean_score = sub[col_name].mean(skipna=True)
            n_valid    = sub[col_name].notna().sum()
            pair_scores.append(mean_score)
            pair_details.append(f"ann{suffix_a}-ann{suffix_b}={mean_score:.4f}(n={n_valid})")

            row[f"{metric}_ann{suffix_a}_ann{suffix_b}"] = round(mean_score, 4)
            row[f"{metric}_ann{suffix_a}_ann{suffix_b}_n"] = n_valid

        row[f"{metric}_mean"] = round(np.nanmean(pair_scores), 4)
        print(f"  {level} {metric.upper():8s}: "
              f"{' | '.join(pair_details)} | mean={row[f'{metric}_mean']:.4f}")

    rouge_iaa_results.append(row)
    print()

rouge_iaa_df = pd.DataFrame(rouge_iaa_results)

# Distribution summary per level per metric
print("\n=== ROUGE Distribution Summary per Level ===\n")
for level in ["L2", "L3", "L4"]:
    sub = long_df[long_df["level"] == level]
    print(f"  {level}:")
    for metric in ROUGE_METRICS:
        # Pool all pair scores for this level+metric
        pair_cols = [ROUGE_PAIR_COLS[(f"{a.split('_')[-1]}{b.split('_')[-1]}", metric)]
                     for (a, b) in ANNOTATOR_PAIRS]
        vals = sub[pair_cols].values.flatten()
        vals = vals[~np.isnan(vals)]
        print(f"    {metric.upper():8s}: min={vals.min():.4f}  "
              f"Q1={vals.quantile(0.25) if hasattr(vals, 'quantile') else np.percentile(vals, 25):.4f}  "
              f"median={np.median(vals):.4f}  "
              f"Q3={np.percentile(vals, 75):.4f}  "
              f"max={vals.max():.4f}")
    print()

Computing ROUGE for ann1 vs ann2 ...
Computing ROUGE for ann1 vs ann3 ...
Computing ROUGE for ann2 vs ann3 ...

Done. Sample:
                                         unit_id level  rouge_rouge1_12  rouge_rouge2_12  rouge_rougeL_12  rouge_rouge1_13  rouge_rouge2_13  rouge_rougeL_13
f451d481-b828-40ed-a9a4-fc7fd6d6caf2__chart1__L2    L2         0.727273         0.285714         0.590909         0.700000         0.315789         0.600000
f451d481-b828-40ed-a9a4-fc7fd6d6caf2__chart1__L3    L3              NaN              NaN              NaN              NaN              NaN              NaN
f451d481-b828-40ed-a9a4-fc7fd6d6caf2__chart1__L4    L4         0.206897         0.000000         0.137931         0.257143         0.000000         0.171429
f451d481-b828-40ed-a9a4-fc7fd6d6caf2__chart2__L2    L2         0.352941         0.000000         0.235294         0.378378         0.000000         0.270270
f451d481-b828-40ed-a9a4-fc7fd6d6caf2__chart2__L3    L3         0.367816         0.117647 

In [22]:
# Bottom 3 units per level by mean ROUGE-L (most discriminative)
print("\n=== Lowest ROUGE-L Units per Level ===\n")

for level in ["L2", "L3", "L4"]:
    sub = long_df[long_df["level"] == level].copy()

    rougeL_cols = [ROUGE_PAIR_COLS[(f"{a.split('_')[-1]}{b.split('_')[-1]}", "rougeL")]
                   for (a, b) in ANNOTATOR_PAIRS]
    sub["mean_rougeL"] = sub[rougeL_cols].mean(axis=1, skipna=True)
    sub_valid = sub[sub["mean_rougeL"].notna()].sort_values("mean_rougeL")

    print(f"{'='*70}")
    print(f"LEVEL {level} — Bottom 3 by mean ROUGE-L")
    print(f"{'='*70}")

    for _, row in sub_valid.head(3).iterrows():
        print(f"\nunit : {row['unit_id']}")
        print(f"chart: {row['chart_id']} — {row['title']}")
        print(f"ROUGE-L: {row['mean_rougeL']:.4f}")
        print(f"ann1 : {row['annotator_1']}")
        print(f"ann2 : {row['annotator_2']}")
        print(f"ann3 : {row['annotator_3']}")


=== Lowest ROUGE-L Units per Level ===

LEVEL L2 — Bottom 3 by mean ROUGE-L

unit : 511bca34-e364-49d6-8cf9-263c3426db13__chart2__L2
chart: 2 — Profit Margin by State
ROUGE-L: 0.0969
ann1 : The majority of states show a positive margin of green encoded in 2021, compared to 2020
ann2 : AK, WY, HI, and ME had N/A value
ann3 : States with negatives prorit margin are OR, Az, Co, IL, Tx, TN, OH, PA and NC while others in green has positive profit margin

unit : dc09d834-178b-48e2-990d-94464475d0fa__chart5__L2
chart: 5 — Monthly Orders Details
ROUGE-L: 0.1174
ann1 : The highest orders occurred in November 2019 at around 250 orders
ann2 : Ranging from 2016 to 2019, almost every month in every year, the number of the later year is more than the earlier
ann3 : 2019 has the biggest monthly orders compared with 2018, 2017, and 2016

unit : b164ac02-5f8b-4b10-ad35-afa018951888__chart2__L2
chart: 2 — 2023 | Sales vs Targets
ROUGE-L: 0.1536
ann1 : Sales started at around £40K in January and ended t

### BLEU Metric

In [23]:
smoother = SmoothingFunction().method1

def bleu_pairwise_batch(refs, hyps):
    assert len(refs) == len(hyps)
    results = {
        "bleu1": np.full(len(refs), np.nan),
        "bleu2": np.full(len(refs), np.nan),
        "bleu4": np.full(len(refs), np.nan),
    }
    for i, (r, h) in enumerate(zip(refs, hyps)):
        if is_missing(r) or is_missing(h):
            continue
        ref_tokens = nltk.word_tokenize(str(r).lower())
        hyp_tokens = nltk.word_tokenize(str(h).lower())
        results["bleu1"][i] = sentence_bleu(
            [ref_tokens], hyp_tokens,
            weights=(1, 0, 0, 0),
            smoothing_function=smoother
        )
        results["bleu2"][i] = sentence_bleu(
            [ref_tokens], hyp_tokens,
            weights=(0.5, 0.5, 0, 0),
            smoothing_function=smoother
        )
        results["bleu4"][i] = sentence_bleu(
            [ref_tokens], hyp_tokens,
            weights=(0.25, 0.25, 0.25, 0.25),
            smoothing_function=smoother
        )
    return results

BLEU_METRICS = ["bleu1", "bleu2", "bleu4"]
BLEU_PAIR_COLS = {}

for (col_a, col_b) in ANNOTATOR_PAIRS:
    suffix_a = col_a.split("_")[-1]
    suffix_b = col_b.split("_")[-1]
    pair_key = f"{suffix_a}{suffix_b}"
    print(f"Computing BLEU for ann{suffix_a} vs ann{suffix_b} ...")
    bleu_results = bleu_pairwise_batch(
        long_df[col_a].tolist(),
        long_df[col_b].tolist()
    )
    for metric in BLEU_METRICS:
        col_name = f"bleu_{metric}_{pair_key}"
        long_df[col_name] = bleu_results[metric]
        BLEU_PAIR_COLS[(pair_key, metric)] = col_name

print("\n=== Mean Pairwise BLEU by Level ===\n")
bleu_iaa_results = []
for level in ["L2", "L3", "L4"]:
    sub = long_df[long_df["level"] == level]
    row = {"Level": level}
    for metric in BLEU_METRICS:
        pair_scores = []
        pair_details = []
        for (col_a, col_b) in ANNOTATOR_PAIRS:
            suffix_a = col_a.split("_")[-1]
            suffix_b = col_b.split("_")[-1]
            pair_key = f"{suffix_a}{suffix_b}"
            col_name = BLEU_PAIR_COLS[(pair_key, metric)]
            mean_score = sub[col_name].mean(skipna=True)
            n_valid    = sub[col_name].notna().sum()
            pair_scores.append(mean_score)
            pair_details.append(f"ann{suffix_a}-ann{suffix_b}={mean_score:.4f}(n={n_valid})")
            row[f"{metric}_ann{suffix_a}_ann{suffix_b}"] = round(mean_score, 4)
        row[f"{metric}_mean"] = round(np.nanmean(pair_scores), 4)
        print(f"  {level} {metric.upper():6s}: "
              f"{' | '.join(pair_details)} | mean={row[f'{metric}_mean']:.4f}")
    bleu_iaa_results.append(row)
    print()

bleu_iaa_df = pd.DataFrame(bleu_iaa_results)

print("\n=== BLEU Distribution Summary per Level ===\n")
for level in ["L2", "L3", "L4"]:
    sub = long_df[long_df["level"] == level]
    print(f"  {level}:")
    for metric in BLEU_METRICS:
        pair_cols = [BLEU_PAIR_COLS[(f"{a.split('_')[-1]}{b.split('_')[-1]}", metric)]
                     for (a, b) in ANNOTATOR_PAIRS]
        vals = sub[pair_cols].values.flatten()
        vals = vals[~np.isnan(vals)]
        print(f"    {metric.upper():6s}: min={vals.min():.4f}  "
              f"median={np.median(vals):.4f}  "
              f"max={vals.max():.4f}")
    print()

Computing BLEU for ann1 vs ann2 ...
Computing BLEU for ann1 vs ann3 ...
Computing BLEU for ann2 vs ann3 ...

=== Mean Pairwise BLEU by Level ===

  L2 BLEU1 : ann1-ann2=0.3771(n=49) | ann1-ann3=0.3477(n=49) | ann2-ann3=0.3973(n=49) | mean=0.3740
  L2 BLEU2 : ann1-ann2=0.2327(n=49) | ann1-ann3=0.1960(n=49) | ann2-ann3=0.2366(n=49) | mean=0.2218
  L2 BLEU4 : ann1-ann2=0.0847(n=49) | ann1-ann3=0.0829(n=49) | ann2-ann3=0.0936(n=49) | mean=0.0871

  L3 BLEU1 : ann1-ann2=0.2426(n=35) | ann1-ann3=0.2027(n=35) | ann2-ann3=0.2770(n=45) | mean=0.2408
  L3 BLEU2 : ann1-ann2=0.1098(n=35) | ann1-ann3=0.0953(n=35) | ann2-ann3=0.1547(n=45) | mean=0.1199
  L3 BLEU4 : ann1-ann2=0.0379(n=35) | ann1-ann3=0.0299(n=35) | ann2-ann3=0.0500(n=45) | mean=0.0392

  L4 BLEU1 : ann1-ann2=0.1747(n=49) | ann1-ann3=0.1851(n=49) | ann2-ann3=0.2430(n=49) | mean=0.2009
  L4 BLEU2 : ann1-ann2=0.0448(n=49) | ann1-ann3=0.0579(n=49) | ann2-ann3=0.1016(n=49) | mean=0.0681
  L4 BLEU4 : ann1-ann2=0.0135(n=49) | ann1-ann3=0.01

### METEOR Metric

In [24]:
def meteor_pairwise_batch(refs, hyps):
    assert len(refs) == len(hyps)
    results = {"meteor": np.full(len(refs), np.nan)}
    for i, (r, h) in enumerate(zip(refs, hyps)):
        if is_missing(r) or is_missing(h):
            continue
        ref_tokens = nltk.word_tokenize(str(r).lower())
        hyp_tokens = nltk.word_tokenize(str(h).lower())
        results["meteor"][i] = meteor_score([ref_tokens], hyp_tokens)
    return results

METEOR_PAIR_COLS = {}

for (col_a, col_b) in ANNOTATOR_PAIRS:
    suffix_a = col_a.split("_")[-1]
    suffix_b = col_b.split("_")[-1]
    pair_key = f"{suffix_a}{suffix_b}"
    print(f"Computing METEOR for ann{suffix_a} vs ann{suffix_b} ...")
    meteor_results = meteor_pairwise_batch(
        long_df[col_a].tolist(),
        long_df[col_b].tolist()
    )
    col_name = f"meteor_{pair_key}"
    long_df[col_name] = meteor_results["meteor"]
    METEOR_PAIR_COLS[pair_key] = col_name

print("\n=== Mean Pairwise METEOR by Level ===\n")
meteor_iaa_results = []
for level in ["L2", "L3", "L4"]:
    sub = long_df[long_df["level"] == level]
    row = {"Level": level}
    pair_scores = []
    pair_details = []
    for (col_a, col_b) in ANNOTATOR_PAIRS:
        suffix_a = col_a.split("_")[-1]
        suffix_b = col_b.split("_")[-1]
        pair_key = f"{suffix_a}{suffix_b}"
        col_name = METEOR_PAIR_COLS[pair_key]
        mean_score = sub[col_name].mean(skipna=True)
        n_valid    = sub[col_name].notna().sum()
        pair_scores.append(mean_score)
        pair_details.append(f"ann{suffix_a}-ann{suffix_b}={mean_score:.4f}(n={n_valid})")
        row[f"meteor_ann{suffix_a}_ann{suffix_b}"] = round(mean_score, 4)
    row["meteor_mean"] = round(np.nanmean(pair_scores), 4)
    print(f"  {level} METEOR: "
          f"{' | '.join(pair_details)} | mean={row['meteor_mean']:.4f}")
    meteor_iaa_results.append(row)

meteor_iaa_df = pd.DataFrame(meteor_iaa_results)

print("\n=== METEOR Distribution Summary per Level ===\n")
for level in ["L2", "L3", "L4"]:
    sub = long_df[long_df["level"] == level]
    meteor_cols = [METEOR_PAIR_COLS[f"{a.split('_')[-1]}{b.split('_')[-1]}"]
                   for (a, b) in ANNOTATOR_PAIRS]
    vals = sub[meteor_cols].values.flatten()
    vals = vals[~np.isnan(vals)]
    print(f"  {level} METEOR: min={vals.min():.4f}  "
          f"median={np.median(vals):.4f}  "
          f"max={vals.max():.4f}")

Computing METEOR for ann1 vs ann2 ...
Computing METEOR for ann1 vs ann3 ...
Computing METEOR for ann2 vs ann3 ...

=== Mean Pairwise METEOR by Level ===

  L2 METEOR: ann1-ann2=0.3612(n=49) | ann1-ann3=0.3196(n=49) | ann2-ann3=0.3867(n=49) | mean=0.3559
  L3 METEOR: ann1-ann2=0.2080(n=35) | ann1-ann3=0.2050(n=35) | ann2-ann3=0.2920(n=45) | mean=0.2350
  L4 METEOR: ann1-ann2=0.1465(n=49) | ann1-ann3=0.1790(n=49) | ann2-ann3=0.2412(n=49) | mean=0.1889

=== METEOR Distribution Summary per Level ===

  L2 METEOR: min=0.0305  median=0.3356  max=0.9906
  L3 METEOR: min=0.0664  median=0.2226  max=0.5443
  L4 METEOR: min=0.0152  median=0.1760  max=0.5556


### IAA Metric Comparison

In [25]:
print("=== Full IAA Metric Comparison (mean across annotator pairs) ===\n")
print(f"{'Level':6} {'BLEU-1':8} {'BLEU-2':8} {'BLEU-4':8} "
      f"{'METEOR':8} {'ROUGE-1':8} {'ROUGE-2':8} {'ROUGE-L':8} {'BERTScore':10}")
print("-" * 78)

for level in ["L2", "L3", "L4"]:
    b  = bleu_iaa_df[bleu_iaa_df["Level"] == level].iloc[0]
    m  = meteor_iaa_df[meteor_iaa_df["Level"] == level].iloc[0]
    r  = rouge_iaa_df[rouge_iaa_df["Level"] == level].iloc[0]
    bs = iaa_df[iaa_df["Level"] == level].iloc[0]
    print(f"{level:6} "
          f"{b['bleu1_mean']:8.4f} "
          f"{b['bleu2_mean']:8.4f} "
          f"{b['bleu4_mean']:8.4f} "
          f"{m['meteor_mean']:8.4f} "
          f"{r['rouge1_mean']:8.4f} "
          f"{r['rouge2_mean']:8.4f} "
          f"{r['rougeL_mean']:8.4f} "
          f"{bs['Mean_F1_overall']:10.4f}")

=== Full IAA Metric Comparison (mean across annotator pairs) ===

Level  BLEU-1   BLEU-2   BLEU-4   METEOR   ROUGE-1  ROUGE-2  ROUGE-L  BERTScore 
------------------------------------------------------------------------------
L2       0.3740   0.2218   0.0871   0.3559   0.4977   0.2088   0.3998     0.4247
L3       0.2408   0.1199   0.0392   0.2350   0.3007   0.0705   0.2147     0.2327
L4       0.2009   0.0681   0.0189   0.1889   0.2518   0.0343   0.1548     0.2010
